# ⚛️ Módulo 5: Dinámica Molecular
## Actividad 5.8: Práctica Completa con GROMACS en Google Colab

<div align="center">

**Universidad de Caldas - Departamento de Química**
*Introducción a la Química Computacional (173G7G)*
**Profesor:** José Mauricio Rodas Rodríguez

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maurorodas/Quimica_computacional_173G7G/blob/main/modulo_05_dinamica_molecular/08_practica_gromacs_openmm.ipynb)

</div>

---

## 🎯 Objetivos de Aprendizaje

Al finalizar esta actividad, serás capaz de:
- Instalar y ejecutar **GROMACS real** en Google Colab mediante `condacolab`
- Ejecutar un protocolo completo de DM: EM → NVT → NPT → Producción
- Preparar sistemas proteína + agua + iones con las herramientas de GROMACS
- Analizar trayectorias con `gmx rms`, `rmsf`, `gyrate` y `energy`
- Interpretar gráficas de RMSD, RMSF, radio de giro y propiedades termodinámicas

---

## ⚠️ Instrucciones Importantes

Este notebook ejecuta **GROMACS real** en Google Colab mediante `condacolab`.

> **Orden de ejecución:**
> 1. Ejecuta la **Celda 1** (instala condacolab) → el kernel se reiniciará automáticamente.
> 2. Tras el reinicio, ejecuta **desde la Celda 2** en adelante.
> 3. No vuelvas a ejecutar la Celda 1.

> 💡 **Tiempo estimado total:** ~60-90 min en CPU. Usa **Entorno de ejecución → GPU** para acelerar.

---

## 1. Instalación: condacolab + GROMACS

In [ ]:
# ──────────────────────────────────────────────────────────────────
# CELDA 1 — Ejecutar UNA sola vez. El kernel se reiniciará.
# ──────────────────────────────────────────────────────────────────
!pip install -q condacolab
import condacolab
condacolab.install()   # ← el kernel se reinicia aquí automáticamente

In [ ]:
# ──────────────────────────────────────────────────────────────────
# CELDA 2 — Instalar GROMACS y paquetes de análisis (tras reinicio)
# ──────────────────────────────────────────────────────────────────
import condacolab
condacolab.check()   # confirma que conda está activo

print("⏳ Instalando GROMACS (conda-forge)... ~5-10 min")
!conda install -c conda-forge -y gromacs 2>&1 | tail -6

print("⏳ Instalando paquetes de análisis...")
!pip install -q MDAnalysis matplotlib numpy pandas scipy seaborn requests

print("✓ Instalación completada")

In [ ]:
import subprocess, os
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path

# Verificar GROMACS
r = subprocess.run(['gmx', '--version'], capture_output=True, text=True)
version_line = [l for l in r.stdout.split('\n') if 'GROMACS version' in l]
if version_line:
    print(f"✓ {version_line[0].strip()}")
else:
    print("✗ GROMACS no encontrado — ejecuta las celdas 1 y 2 primero")

# Verificar GPU — REQUERIDA para este notebook
gpu = subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True)
if gpu.returncode == 0:
    print(f"✓ GPU disponible: {gpu.stdout.strip()}")
else:
    print("⚠️  Sin GPU detectada.")
    print("    → Ve a: Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU (T4)")
    print("    Las celdas de mdrun fallarán sin GPU.")

# Crear directorio de trabajo y moverse a él
os.makedirs('/content/ubq_sim', exist_ok=True)
%cd /content/ubq_sim
print(f"\n✓ Directorio de trabajo: {os.getcwd()}")

## 2. Flujo de Trabajo con GROMACS

```
PDB → pdb2gmx → editconf → solvate → genion → EM → NVT → NPT → MD → Análisis
```

| Etapa | Herramienta | Archivo de salida |
|---|---|---|
| Topología | `gmx pdb2gmx` | `processed.gro`, `topol.top`, `posre.itp` |
| Caja | `gmx editconf` | `newbox.gro` |
| Solvatación | `gmx solvate` | `solv.gro` |
| Iones | `gmx genion` | `solv_ions.gro` |
| Minimización | `gmx mdrun` | `em.gro`, `em.edr` |
| NVT 50 ps | `gmx mdrun` | `nvt.gro`, `nvt.cpt` |
| NPT 50 ps | `gmx mdrun` | `npt.gro`, `npt.cpt` |
| Producción 100 ps | `gmx mdrun` | `md_0_1.xtc`, `md_0_1.edr` |
| Análisis | `gmx rms`, `rmsf`, `gyrate`, `energy` | `.xvg` → gráficas |

**Sistema:** Ubiquitina humana (PDB: 1UBQ, 76 residuos) | **Campo de fuerza:** AMBER99SB-ILDN + SPC/E | **Caja:** cúbica 1.0 nm de margen

## 3. Descarga y Preparación de la Proteína

**Ubiquitina humana (1UBQ):** 76 residuos, 602 átomos de proteína.  
Se eliminan las moléculas de agua cristalográfica (HETATM) porque GROMACS añade agua explícita de manera controlada.

In [ ]:
import requests

# Descargar 1UBQ desde RCSB PDB
url = 'https://files.rcsb.org/download/1UBQ.pdb'
r = requests.get(url, timeout=30)
r.raise_for_status()
with open('1UBQ.pdb', 'w') as f:
    f.write(r.text)
print(f"✓ Descargado 1UBQ.pdb ({len(r.text):,} bytes)")

# Limpiar PDB: conservar solo líneas ATOM (quitar HETATM = agua cristalográfica)
with open('1UBQ.pdb') as f:
    lines = f.readlines()

clean = [l for l in lines if l.startswith('ATOM')]
clean.append('END\n')

with open('1UBQ_clean.pdb', 'w') as f:
    f.writelines(clean)

n_atoms = sum(1 for l in clean if l.startswith('ATOM'))
n_residues = len(set(l[22:26].strip() for l in clean if l.startswith('ATOM')))
print(f"✓ 1UBQ_clean.pdb: {n_residues} residuos, {n_atoms} átomos de proteína")

## 4. Preparación de la Topología (`pdb2gmx`)

`pdb2gmx` genera:
- `processed.gro` — coordenadas en formato GROMACS (con H añadidos)
- `topol.top` — topología completa del sistema
- `posre.itp` — restricciones de posición para la equilibración

In [ ]:
# Campo de fuerza: AMBER99SB-ILDN | Modelo de agua: SPC/E
# -ignh: ignorar hidrógenos del PDB (GROMACS añadirá los correctos)
!gmx pdb2gmx \
    -f 1UBQ_clean.pdb \
    -o processed.gro \
    -water spce \
    -ff amber99sb-ildn \
    -ignh 2>&1 | tail -20

# Verificar archivos generados
for fname in ['processed.gro', 'topol.top', 'posre.itp']:
    estado = '✓' if os.path.exists(fname) else '✗ FALTA'
    print(f"  {estado} {fname}")

## 5. Construcción del Sistema: Caja, Solvatación e Iones

| Paso | Comando | Descripción |
|---|---|---|
| 5.1 | `editconf` | Define caja cúbica con 1.0 nm de margen |
| 5.2 | `solvate` | Rellena la caja con agua SPC/E |
| 5.3 | `genion` | Añade Na⁺/Cl⁻ para neutralizar la carga |

In [ ]:
# 5.1 Caja cúbica — 1.0 nm de margen alrededor de la proteína
print("--- editconf ---")
!gmx editconf -f processed.gro -o newbox.gro -c -d 1.0 -bt cubic 2>&1 | tail -5

# 5.2 Solvatación con SPC/E
print("\n--- solvate ---")
!gmx solvate -cp newbox.gro -cs spc216.gro -o solv.gro -p topol.top 2>&1 | tail -5

# 5.3 Preparar tpr mínimo para genion
ions_mdp = """; ions.mdp — parámetros mínimos para selección de grupos
integrator  = steep
emtol       = 1000.0
nsteps      = 50000
nstlist     = 1
cutoff-scheme = Verlet
rcoulomb    = 1.0
rvdw        = 1.0
pbc         = xyz
"""
with open('ions.mdp', 'w') as f:
    f.write(ions_mdp)

print("\n--- grompp (ions) ---")
!gmx grompp -f ions.mdp -c solv.gro -p topol.top -o ions.tpr -maxwarn 1 2>&1 | tail -5

# 5.4 Añadir iones — seleccionar grupo SOL para reemplazar
print("\n--- genion ---")
!printf "SOL\n" | gmx genion -s ions.tpr -o solv_ions.gro \
    -p topol.top -pname NA -nname CL -neutral 2>&1 | tail -8

# Verificar
for fname in ['newbox.gro', 'solv.gro', 'solv_ions.gro']:
    estado = '✓' if os.path.exists(fname) else '✗'
    print(f"  {estado} {fname}")

## 6. Minimización de Energía

Elimina contactos desfavorables (distancias incorrectas, choques estéricos) antes de la dinámica.  
Criterio de convergencia: fuerza máxima < 1000 kJ·mol⁻¹·nm⁻¹.

In [ ]:
minim_mdp = """; minim.mdp — minimización de energía (steepest descent)
integrator  = steep
emtol       = 1000.0
emstep      = 0.01
nsteps      = 50000
nstlist         = 10
cutoff-scheme   = Verlet
ns_type         = grid
coulombtype     = PME
rcoulomb        = 1.0
rvdw            = 1.0
pbc             = xyz
"""
with open('minim.mdp', 'w') as f:
    f.write(minim_mdp)

print("--- grompp (EM) ---")
!gmx grompp -f minim.mdp -c solv_ions.gro -p topol.top -o em.tpr -maxwarn 1 2>&1 | tail -5

print("\n⏳ Minimizando energía (~1-2 min en GPU)...")
# EM solo soporta -nb gpu (sin -pme gpu ni -update gpu)
!gmx mdrun -v -deffnm em -ntmpi 1 -nb gpu 2>&1 | tail -12

if os.path.exists('em.gro'):
    print("\n✓ Minimización completada: em.gro")
    !printf "Potential\n0\n" | gmx energy -f em.edr -o em_potential.xvg 2>&1 | grep "Potential"
else:
    print("✗ Error en minimización")

## 7. Equilibración NVT (50 ps, 300 K)

Se aplican **restricciones de posición** sobre la proteína (`-DPOSRES`) para que el solvente se equilibre alrededor de la estructura fija.  
Termostato: V-rescale (τ = 0.1 ps).

In [ ]:
nvt_mdp = """; nvt.mdp — equilibración NVT 50 ps
title                   = NVT equilibration
define                  = -DPOSRES
integrator              = md
nsteps                  = 25000
dt                      = 0.002
nstxout                 = 500
nstvout                 = 500
nstenergy               = 500
nstlog                  = 500
nstxout-compressed      = 500
continuation            = no
constraint_algorithm    = lincs
constraints             = h-bonds
lincs_iter              = 1
lincs_order             = 4
cutoff-scheme           = Verlet
ns_type                 = grid
nstlist                 = 20
rcoulomb                = 1.0
rvdw                    = 1.0
coulombtype             = PME
pme_order               = 4
fourierspacing          = 0.16
tcoupl                  = V-rescale
tc-grps                 = Protein Non-Protein
tau_t                   = 0.1   0.1
ref_t                   = 300   300
pcoupl                  = no
pbc                     = xyz
DispCorr                = EnerPres
"""
with open('nvt.mdp', 'w') as f:
    f.write(nvt_mdp)

print("--- grompp (NVT) ---")
!gmx grompp -f nvt.mdp -c em.gro -r em.gro -p topol.top -o nvt.tpr -maxwarn 1 2>&1 | tail -5

print("\n⏳ Equilibración NVT (~2-5 min en GPU)...")
!gmx mdrun -deffnm nvt -ntmpi 1 -nb gpu -pme gpu 2>&1 | tail -10

if os.path.exists('nvt.gro'):
    print("\n✓ NVT completada: nvt.gro, nvt.cpt")
else:
    print("✗ Error en NVT")

## 8. Equilibración NPT (50 ps, 300 K, 1 bar)

Se mantienen las restricciones de posición y se activa el barostato Berendsen.  
Al finalizar, la densidad del sistema debe converger a ~1.0 g/mL.

In [ ]:
npt_mdp = """; npt.mdp — equilibración NPT 50 ps
title                   = NPT equilibration
define                  = -DPOSRES
integrator              = md
nsteps                  = 25000
dt                      = 0.002
nstxout                 = 500
nstvout                 = 500
nstenergy               = 500
nstlog                  = 500
nstxout-compressed      = 500
continuation            = yes
constraint_algorithm    = lincs
constraints             = h-bonds
lincs_iter              = 1
lincs_order             = 4
cutoff-scheme           = Verlet
ns_type                 = grid
nstlist                 = 20
rcoulomb                = 1.0
rvdw                    = 1.0
coulombtype             = PME
pme_order               = 4
fourierspacing          = 0.16
tcoupl                  = V-rescale
tc-grps                 = Protein Non-Protein
tau_t                   = 0.1   0.1
ref_t                   = 300   300
pcoupl                  = Berendsen
pcoupltype              = isotropic
tau_p                   = 2.0
ref_p                   = 1.0
compressibility         = 4.5e-5
refcoord_scaling        = com
pbc                     = xyz
DispCorr                = EnerPres
"""
with open('npt.mdp', 'w') as f:
    f.write(npt_mdp)

print("--- grompp (NPT) ---")
!gmx grompp -f npt.mdp -c nvt.gro -r nvt.gro -t nvt.cpt \
    -p topol.top -o npt.tpr -maxwarn 1 2>&1 | tail -5

print("\n⏳ Equilibración NPT (~2-5 min en GPU)...")
!gmx mdrun -deffnm npt -ntmpi 1 -nb gpu -pme gpu 2>&1 | tail -10

if os.path.exists('npt.gro'):
    print("\n✓ NPT completada: npt.gro, npt.cpt")
    !printf "Density\n0\n" | gmx energy -f npt.edr -o density.xvg 2>&1 | grep "Density"
else:
    print("✗ Error en NPT")

## 9. Dinámica Molecular de Producción (100 ps)

Se eliminan las restricciones de posición. Barostato Parrinello-Rahman (más riguroso que Berendsen).  
**Archivo de salida principal:** `md_0_1.xtc` (trayectoria comprimida, ~500 frames).

In [ ]:
md_mdp = """; md.mdp — producción MD 100 ps (sin restricciones de posición)
title                   = Ubiquitina MD production
integrator              = md
nsteps                  = 50000
dt                      = 0.002
nstxout                 = 0
nstvout                 = 0
nstenergy               = 500
nstlog                  = 500
nstxout-compressed      = 200
compressed-x-grps       = System
continuation            = yes
constraint_algorithm    = lincs
constraints             = h-bonds
lincs_iter              = 1
lincs_order             = 4
cutoff-scheme           = Verlet
ns_type                 = grid
nstlist                 = 20
rcoulomb                = 1.0
rvdw                    = 1.0
coulombtype             = PME
pme_order               = 4
fourierspacing          = 0.16
tcoupl                  = V-rescale
tc-grps                 = Protein Non-Protein
tau_t                   = 0.1   0.1
ref_t                   = 300   300
pcoupl                  = Parrinello-Rahman
pcoupltype              = isotropic
tau_p                   = 2.0
ref_p                   = 1.0
compressibility         = 4.5e-5
pbc                     = xyz
DispCorr                = EnerPres
"""
with open('md.mdp', 'w') as f:
    f.write(md_mdp)

print("--- grompp (producción) ---")
!gmx grompp -f md.mdp -c npt.gro -t npt.cpt \
    -p topol.top -o md_0_1.tpr -maxwarn 1 2>&1 | tail -5

print("\n⏳ Dinámica de producción (~5-10 min en GPU)...")
!gmx mdrun -deffnm md_0_1 -ntmpi 1 -nb gpu -pme gpu 2>&1 | tail -15

if os.path.exists('md_0_1.xtc'):
    print("\n✓ Producción completada!")
    print("  md_0_1.xtc — trayectoria")
    print("  md_0_1.edr — datos de energía")
    print("  md_0_1.gro — última estructura")
else:
    print("✗ Error en la producción")

## 10. Análisis de la Trayectoria

### 10.1 Corrección de Condiciones de Contorno Periódicas (PBC)

La proteína puede "saltar" a través de los bordes de la caja durante la simulación. `trjconv` la centra y reconstruye la imagen continua.

In [ ]:
# Corrección PBC: centrar proteína y hacer la imagen continua
# Grupo 1 = Protein (centrar), Grupo 2 = System (salida)
!printf "Protein\nSystem\n" | gmx trjconv \
    -s md_0_1.tpr \
    -f md_0_1.xtc \
    -o md_noPBC.xtc \
    -pbc mol \
    -center 2>&1 | tail -5

print("✓ Trayectoria corregida: md_noPBC.xtc")

### 10.2 Análisis Estructural: RMSD, RMSF y Radio de Giro

In [ ]:
# RMSD del backbone respecto a la estructura inicial (em.gro)
print("--- RMSD ---")
!printf "Backbone\nBackbone\n" | gmx rms \
    -s md_0_1.tpr \
    -f md_noPBC.xtc \
    -o rmsd.xvg \
    -tu ps 2>&1 | tail -4

# RMSF por residuo (fluctuaciones de los Cα)
print("--- RMSF ---")
!printf "C-alpha\n" | gmx rmsf \
    -s md_0_1.tpr \
    -f md_noPBC.xtc \
    -o rmsf.xvg \
    -res 2>&1 | tail -4

# Radio de giro
print("--- Gyrate ---")
!printf "Protein\n" | gmx gyrate \
    -s md_0_1.tpr \
    -f md_noPBC.xtc \
    -o gyrate.xvg 2>&1 | tail -4

print("\n✓ Archivos de análisis generados: rmsd.xvg, rmsf.xvg, gyrate.xvg")

In [ ]:
def leer_xvg(archivo):
    """Lee archivos .xvg de GROMACS, omite líneas de comentario (# y @)."""
    datos = []
    with open(archivo) as f:
        for linea in f:
            linea = linea.strip()
            if linea and not linea.startswith(('#', '@')):
                try:
                    datos.append([float(x) for x in linea.split()])
                except ValueError:
                    pass
    import numpy as np
    return np.array(datos)

# ── Graficar RMSD, RMSF y Radio de Giro ──────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# RMSD (nm → Å multiplicando por 10)
rmsd = leer_xvg('rmsd.xvg')
axes[0].plot(rmsd[:, 0], rmsd[:, 1] * 10, color='steelblue', linewidth=1.5)
axes[0].set_xlabel('Tiempo (ps)', fontsize=12)
axes[0].set_ylabel('RMSD (Å)', fontsize=12)
axes[0].set_title('RMSD del Backbone', fontsize=13)
axes[0].grid(True, alpha=0.3)

# RMSF por residuo
rmsf = leer_xvg('rmsf.xvg')
axes[1].bar(rmsf[:, 0], rmsf[:, 1] * 10,
            color='seagreen', alpha=0.75, width=0.8)
axes[1].set_xlabel('Número de Residuo', fontsize=12)
axes[1].set_ylabel('RMSF (Å)', fontsize=12)
axes[1].set_title('Fluctuaciones por Residuo (RMSF)', fontsize=13)
axes[1].grid(True, alpha=0.3, axis='y')

# Radio de Giro
gy = leer_xvg('gyrate.xvg')
axes[2].plot(gy[:, 0], gy[:, 1], color='firebrick', linewidth=1.5)
axes[2].set_xlabel('Tiempo (ps)', fontsize=12)
axes[2].set_ylabel('Rg (nm)', fontsize=12)
axes[2].set_title('Radio de Giro', fontsize=13)
axes[2].grid(True, alpha=0.3)

plt.suptitle('Análisis Estructural — Ubiquitina (1UBQ)', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('analisis_estructural.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Guardado: analisis_estructural.png")

### 10.3 Análisis Termodinámico: Temperatura, Presión y Energía Potencial

In [ ]:
# Extraer propiedades del archivo de energía (.edr)
print("Extrayendo propiedades termodinámicas...")
!printf "Temperature\n0\n" | gmx energy -f md_0_1.edr -o temperature.xvg 2>&1 | grep -i "temperature\|mean\|std"
!printf "Pressure\n0\n"    | gmx energy -f md_0_1.edr -o pressure.xvg    2>&1 | grep -i "pressure\|mean\|std"
!printf "Potential\n0\n"   | gmx energy -f md_0_1.edr -o potential.xvg   2>&1 | grep -i "potential\|mean\|std"
!printf "Density\n0\n"     | gmx energy -f md_0_1.edr -o density_md.xvg  2>&1 | grep -i "density\|mean\|std"

# ── Graficar propiedades termodinámicas ───────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(13, 8))
axes = axes.flatten()

propiedades = [
    ('temperature.xvg', 'Temperatura (K)',          'navy',      'Temperatura'),
    ('pressure.xvg',    'Presión (bar)',             'darkorange','Presión'),
    ('potential.xvg',   'Energía Potencial (kJ/mol)','crimson',   'Energía Potencial'),
    ('density_md.xvg',  'Densidad (kg/m³)',          'purple',    'Densidad'),
]

for ax, (archivo, ylabel, color, titulo) in zip(axes, propiedades):
    try:
        d = leer_xvg(archivo)
        ax.plot(d[:, 0], d[:, 1], color=color, linewidth=1.0, alpha=0.85)
        media = d[:, 1].mean()
        ax.axhline(media, color='k', linewidth=1.2, linestyle='--',
                   label=f'Media: {media:.1f}')
        ax.set_xlabel('Tiempo (ps)', fontsize=11)
        ax.set_ylabel(ylabel, fontsize=11)
        ax.set_title(titulo, fontsize=12)
        ax.legend(fontsize=10)
        ax.grid(True, alpha=0.3)
    except Exception as e:
        ax.text(0.5, 0.5, f'Error al leer\n{archivo}',
                transform=ax.transAxes, ha='center', va='center', color='red')

plt.suptitle('Propiedades Termodinámicas — Producción MD (100 ps)', fontsize=14)
plt.tight_layout()
plt.savefig('analisis_termodinamico.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Guardado: analisis_termodinamico.png")

### 10.4 Resumen de Archivos Generados

In [ ]:
import os

archivos_clave = {
    'Proteína original':    '1UBQ.pdb',
    'Proteína limpia':      '1UBQ_clean.pdb',
    'Topología':            'topol.top',
    'Restricciones posición':'posre.itp',
    'Sistema solvatado':    'solv_ions.gro',
    'Estructura EM':        'em.gro',
    'Estructura NVT':       'nvt.gro',
    'Estructura NPT':       'npt.gro',
    'Trayectoria raw':      'md_0_1.xtc',
    'Trayectoria corregida':'md_noPBC.xtc',
    'Energías MD':          'md_0_1.edr',
    'RMSD':                 'rmsd.xvg',
    'RMSF':                 'rmsf.xvg',
    'Radio de Giro':        'gyrate.xvg',
    'Gráfica estructural':  'analisis_estructural.png',
    'Gráfica termodinám.':  'analisis_termodinamico.png',
}

print(f"{'Archivo':<25} {'Estado':<8} {'Tamaño':>10}")
print("-" * 46)
for desc, fname in archivos_clave.items():
    if os.path.exists(fname):
        size = os.path.getsize(fname)
        size_str = f"{size/1024:.1f} KB" if size < 1e6 else f"{size/1e6:.1f} MB"
        print(f"{fname:<25} {'✓':<8} {size_str:>10}")
    else:
        print(f"{fname:<25} {'✗ falta':<8} {'':>10}")

## 11. Proyecto Final del Módulo

Repite el flujo completo con **una proteína de tu elección** y entrega un informe que incluya:

1. **Introducción** — importancia biológica de la proteína
2. **Metodología** — campo de fuerza, caja, protocolo
3. **Resultados** — RMSD, RMSF, Rg, temperatura, densidad, energía
4. **Conclusiones** — observación biológicamente relevante

### Proteínas sugeridas

| Proteína | PDB | Residuos | Tiempo sugerido |
|---|---|---|---|
| Trp-cage | 1L2Y | 20 | 200 ps |
| Villin headpiece | 1VII | 35 | 200 ps |
| Ubiquitina | 1UBQ | 76 | 100 ps *(este tutorial)* |
| Lisozima de huevo | 1AKI | 129 | 200 ps |
| DHFR | 1DRF | 186 | 500 ps |

In [ ]:
plantilla_informe = """
# Informe: Simulación de Dinámica Molecular de [NOMBRE_PROTEÍNA]

**Autor:** [Tu nombre]
**Fecha:** [Fecha]
**Código PDB:** [XXXX]

---

## 1. Introducción
[Describe brevemente la proteína y su importancia biológica]

## 2. Metodología

### Software
- GROMACS [versión]
- Campo de fuerza: AMBER99SB-ILDN
- Modelo de agua: SPC/E

### Sistema
| Parámetro | Valor |
|-----------|-------|
| Residuos | |
| Átomos totales (incl. agua) | |
| Tipo de caja | Cúbica |
| Margen de caja | 1.0 nm |
| Concentración iónica | Neutral (NA/CL) |

### Protocolo
| Etapa | Tiempo | Ensamble | Termostato | Barostato |
|-------|--------|----------|------------|-----------|
| EM | — | — | — | — |
| NVT | 50 ps | NVT | V-rescale | — |
| NPT | 50 ps | NPT | V-rescale | Berendsen |
| Producción | 100 ps | NPT | V-rescale | Parrinello-Rahman |

## 3. Resultados

### 3.1 RMSD — ¿Es estable la proteína?
[Gráfica + comentario]

### 3.2 RMSF — ¿Qué regiones son más flexibles?
[Gráfica + comentario]

### 3.3 Radio de Giro — ¿Mantiene su estructura compacta?
[Gráfica + comentario]

### 3.4 Propiedades Termodinámicas
- Temperatura media: ___ ± ___ K
- Densidad media: ___ kg/m³
- Energía potencial: ___ kJ/mol

## 4. Conclusiones
[Describe los principales hallazgos de la simulación]

## 5. Referencias
1. Lindahl et al. GROMACS 2021 Source code. Zenodo (2021).
2. Lindorff-Larsen et al. Improved side-chain torsion potentials for the Amber ff99SB protein force field. Proteins (2010).
3. [Referencia de la proteína usada]
"""

import os
os.makedirs('/content/ubq_sim', exist_ok=True)
with open('/content/ubq_sim/plantilla_informe.md', 'w') as f:
    f.write(plantilla_informe)
print("✓ Plantilla guardada: /content/ubq_sim/plantilla_informe.md")

## 12. Resumen del Módulo 5

| Actividad | Tema | Herramientas |
|-----------|------|--------------|
| 5.1 | Fundamentos de DM | Python, NumPy, Matplotlib |
| 5.2 | Integradores (Verlet, LF, VV) | Python, NumPy |
| 5.3 | PBC y Ensambles | Python, NumPy |
| 5.4 | Termostatos y Barostatos | Python |
| 5.5 | Preparación de Sistemas | GROMACS, BioPython |
| 5.6 | Simulación de Proteínas | GROMACS |
| 5.7 | Análisis de Trayectorias | MDAnalysis, gmx tools |
| **5.8** | **Práctica Completa GROMACS** | **GROMACS (Colab)** |

### Flujo completo que has ejecutado hoy

```
condacolab → GROMACS → pdb2gmx → editconf → solvate
    → genion → EM → NVT → NPT → MD production
    → trjconv → rms → rmsf → gyrate → energy → gráficas
```

### Próximos pasos
- **Enhanced sampling:** metadinámica, REMD, steered MD
- **Energía libre:** FEP, umbrella sampling, AWH
- **Coarse-graining:** MARTINI
- **Machine learning force fields:** ANI, MACE, NequIP

## 9. Recursos Adicionales

### Software
- [GROMACS](https://www.gromacs.org/) — Motor de DM de alto rendimiento
- [OpenMM](https://openmm.org/) — DM en Python con GPU
- [NAMD](https://www.ks.uiuc.edu/Research/namd/) — DM para sistemas grandes
- [AMBER](https://ambermd.org/) — Suite completa de DM biomolecular
- [VMD](https://www.ks.uiuc.edu/Research/vmd/) — Visualización de trayectorias

### Tutoriales
- [GROMACS Tutorials (Justin Lemkul)](http://www.mdtutorials.com/gmx/)
- [OpenMM User Guide](https://openmm.org/documentation/latest/userguide/)
- [MDAnalysis Tutorials](https://www.mdanalysis.org/MDAnalysisTutorial/)
- [BioExcel Training](https://bioexcel.eu/training/)

### Bases de datos y recursos
- [RCSB PDB](https://www.rcsb.org/) — Estructuras de proteínas
- [CHARMM-GUI](https://www.charmm-gui.org/) — Preparación de sistemas
- [SWISS-MODEL](https://swissmodel.expasy.org/) — Modelado por homología

### Libros de referencia
- Frenkel & Smit, *Understanding Molecular Simulation* (2002)
- Allen & Tildesley, *Computer Simulation of Liquids* (2017)
- Tuckerman, *Statistical Mechanics: Theory and Molecular Simulation* (2010)
- Leach, *Molecular Modelling: Principles and Applications* (2001)

---

## ✅ Verificación de Aprendizaje

Al finalizar esta actividad deberías ser capaz de:

- ✅ Ejecutar de forma autónoma un protocolo completo de DM con GROMACS y/o OpenMM
- ✅ Comparar el desempeño y las capacidades de ambos softwares
- ✅ Analizar e interpretar los resultados de la simulación con herramientas integradas
- ✅ Evaluar la calidad de la simulación y detectar posibles artefactos
- ✅ Generar un informe científico con los resultados del análisis

---

<div align="center">

## 🎉 ¡Felicitaciones!

Has completado la **Actividad 5.8: Práctica con GROMACS y OpenMM**

¡Has finalizado el **Módulo 5: Dinámica Molecular** completo! 🚀

[![Anterior](https://img.shields.io/badge/⬅️_Actividad_5.7-Análisis_de_Trayectorias-blue.svg)](07_analisis_trayectorias.ipynb)

---

📚 **[Volver al Módulo 5](README.md)** | 🏠 **[Inicio del Curso](../README.md)**

---

**Universidad de Caldas - Departamento de Química**  
*Química Computacional 173G7G*

</div>